# ⚙️ Transformación de Datos - Capa Silver

## Objetivo
Este notebook transforma los datos **raw** de la capa Bronze en datos **limpios y tipados** listos para análisis en la capa Silver.

## Arquitectura Medallion - Capa Silver
- **Bronze (Raw)**: Datos sin procesar ✅ Completado
- **Silver (Cleansed)**: Datos limpios, validados y tipados 👈 **Estás aquí**
- **Gold (Curated)**: Datos agregados para análisis y reporting

## Transformaciones Principales

### 1. 🔢 Conversión de Tipos de Datos
Convertir campos de `String` a tipos apropiados:
- Numéricos: `credit_score`, `original_upb`, `original_interest_rate`, `original_ltv`, `original_dti`
- Enteros: `original_loan_term`, `number_of_units`, `number_of_borrowers`

### 2. ✅ Validación y Filtrado
Eliminar registros con:
- Valores nulos en campos críticos (`loan_id`, `original_upb`, `original_ltv`, `original_interest_rate`)
- Valores fuera de rango (ej: `credit_score` debe estar entre 300-850)
- Valores negativos o cero en campos monetarios

### 3. 🔄 Estandarización
Normalizar campos categóricos:
- Convertir a mayúsculas
- Eliminar espacios en blanco
- Consistencia en códigos de estado, tipo de propiedad, etc.

### 4. 💾 Persistencia
Guardar datos transformados en: `credit_risk_platform.silver.loans`

---

## 📚 Paso 1: Leer Datos de la Capa Bronze

### Tabla Fuente
- **Tabla**: `credit_risk_platform.bronze.origination`
- **Formato**: Delta Lake
- **Tipos de datos**: Todos `String` (datos raw sin transformar)

### ¿Por qué leer desde Bronze?
La capa Bronze contiene los datos tal como fueron ingestados, sin transformaciones. Es nuestro punto de partida para la limpieza y tipado.

In [0]:
# ============================================================
# LEER TABLA BRONZE
# ============================================================
# Leer todos los datos de la tabla de originación en Bronze
# Todos los campos son String en este punto
df = spark.read.table("credit_risk_platform.bronze.origination")

print(f"✅ Datos leídos de Bronze: {df.count():,} registros")
print(f"📊 Columnas: {len(df.columns)}")
print(f"\n🔍 Vista previa de los datos:")
display(df.limit(10))

---
## 🔢 Paso 2: Conversión de Tipos de Datos

### ¿Por qué convertir tipos?
En Bronze, todos los campos son `String` para preservar los datos raw. En Silver, necesitamos tipos apropiados para:
- **Análisis numérico**: Cálculos, agregaciones, estadísticas
- **Validación de rangos**: Detectar valores fuera de límites esperados
- **Eficiencia**: Menor uso de memoria y mejor performance

### Conversiones Aplicadas:

#### Campos Numéricos (`double`):
- `credit_score`: Puntuación crediticia (300-850)
- `original_upb`: Saldo principal original en dólares
- `original_ltv`: Loan-to-Value ratio (% del valor de la propiedad)
- `original_cltv`: Combined Loan-to-Value ratio
- `original_interest_rate`: Tasa de interés anual (%)
- `original_dti`: Debt-to-Income ratio (%)

#### Campos Enteros (`int`):
- `original_loan_term`: Plazo del préstamo en meses (ej: 360 = 30 años)
- `number_of_units`: Número de unidades en la propiedad
- `number_of_borrowers`: Número de prestatarios

💡 **Nota**: Si un valor no puede convertirse (ej: "abc" a double), Spark lo convierte a `null`. Estos nulls se manejarán en el paso de validación.

In [0]:
from pyspark.sql.functions import col

# ============================================================
# CONVERTIR TIPOS DE DATOS
# ============================================================
# Convertir campos de String a tipos numéricos apropiados
# Si un valor no puede convertirse, se convierte a NULL

df_typed = df \
    .withColumn("credit_score", col("credit_score").cast("double")) \
    .withColumn("original_upb", col("original_upb").cast("double")) \
    .withColumn("original_ltv", col("original_ltv").cast("double")) \
    .withColumn("original_cltv", col("original_cltv").cast("double")) \
    .withColumn("original_interest_rate", col("original_interest_rate").cast("double")) \
    .withColumn("original_dti", col("original_dti").cast("double")) \
    .withColumn("original_loan_term", col("original_loan_term").cast("int")) \
    .withColumn("number_of_units", col("number_of_units").cast("int")) \
    .withColumn("number_of_borrowers", col("number_of_borrowers").cast("int"))

print("✅ Tipos de datos convertidos exitosamente")
print(f"\n🔍 Vista previa con tipos convertidos:")
display(df_typed.limit(10))

---
## ✅ Paso 3: Validación y Filtrado de Datos

### ¿Por qué validar?
Después de la conversión de tipos, debemos eliminar registros con:
- **Datos faltantes**: Campos críticos con valores `null`
- **Datos inválidos**: Valores fuera de rangos esperados
- **Datos inconsistentes**: Violaciones de reglas de negocio

### Reglas de Validación Aplicadas:

#### 1. Campos Obligatorios (no pueden ser NULL):
- `loan_id`: Identificador único del préstamo
- `original_upb`: Saldo principal original
- `original_ltv`: Loan-to-Value ratio
- `original_interest_rate`: Tasa de interés

#### 2. Validaciones de Rango:
- `original_upb > 0`: El saldo debe ser positivo
- `original_ltv > 0`: LTV debe ser positivo
- `credit_score`: Si existe, debe estar entre 300 y 850

#### 3. Lógica de Filtrado:
```python
# Mantener registros que cumplan TODAS las condiciones:
.filter(loan_id no es NULL)
.filter(original_upb no es NULL Y original_upb > 0)
.filter(original_ltv no es NULL Y original_ltv > 0)
.filter(original_interest_rate no es NULL)
.filter(credit_score es NULL O (credit_score >= 300 Y credit_score <= 850))
```

💡 **Nota sobre credit_score**: Permitimos `NULL` porque algunos préstamos pueden no tener puntuación crediticia disponible, pero si existe, debe estar en el rango válido.

In [0]:
# ============================================================
# APLICAR REGLAS DE VALIDACIÓN Y FILTRADO
# ============================================================
# Eliminar registros que no cumplan con las reglas de negocio

df_filtered = df_typed \
    .filter(col("loan_id").isNotNull()) \
    .filter(col("original_upb").isNotNull() & (col("original_upb") > 0)) \
    .filter(col("original_ltv").isNotNull() & (col("original_ltv") > 0)) \
    .filter(col("original_interest_rate").isNotNull()) \
    .filter(col("credit_score").isNull() | ((col("credit_score") >= 300) & (col("credit_score") <= 850)))

print("✅ Reglas de validación aplicadas")

In [0]:
# ============================================================
# VALIDAR IMPACTO DEL FILTRADO
# ============================================================
# Comparar cantidad de registros antes y después del filtrado

records_before = df_typed.count()
records_after = df_filtered.count()
records_removed = records_before - records_after
removal_pct = (records_removed / records_before) * 100

print("="*60)
print("📊 IMPACTO DEL FILTRADO")
print("="*60)
print(f"\n📄 Registros antes del filtrado: {records_before:,}")
print(f"✅ Registros después del filtrado: {records_after:,}")
print(f"❌ Registros eliminados: {records_removed:,} ({removal_pct:.2f}%)")
print("\n" + "="*60)

---
## 🔄 Paso 4: Estandarización de Campos Categóricos

### ¿Por qué estandarizar?
Los campos categóricos pueden tener inconsistencias:
- **Mayúsculas/minúsculas**: "TX" vs "tx" vs "Tx"
- **Espacios en blanco**: " TX " vs "TX"
- **Formatos mixtos**: Dificultan agrupaciones y joins

### Transformaciones Aplicadas:

#### Función `upper()`:
Convierte texto a mayúsculas
- Ejemplo: "tx" → "TX"

#### Función `trim()`:
Elimina espacios al inicio y final
- Ejemplo: " TX " → "TX"

### Campos Estandarizados:
1. **property_state**: Código del estado (ej: "TX", "CA", "NY")
2. **loan_purpose**: Propósito del préstamo (ej: "P" = Purchase, "R" = Refinance)
3. **property_type**: Tipo de propiedad (ej: "SF" = Single Family, "CO" = Condo)

💡 **Resultado**: Todos los valores categóricos estarán en **mayúsculas** y **sin espacios extras**, facilitando consultas y agrupaciones.

In [0]:
from pyspark.sql.functions import upper, trim

# ============================================================
# ESTANDARIZAR CAMPOS CATEGÓRICOS
# ============================================================
# Convertir a mayúsculas y eliminar espacios en blanco
# Esto garantiza consistencia en códigos y facilita joins/agrupaciones

df_standar = df_filtered \
    .withColumn("property_state", upper(trim(col("property_state")))) \
    .withColumn("loan_purpose", upper(trim(col("loan_purpose")))) \
    .withColumn("property_type", upper(trim(col("property_type"))))

print("✅ Campos categóricos estandarizados (mayúsculas, sin espacios)")

---
## 💾 Paso 5: Guardar en Tabla Silver

### Tabla Destino
- **Nombre**: `credit_risk_platform.silver.loans`
- **Formato**: Delta Lake
- **Modo de escritura**: `overwrite` (reemplazar datos existentes)

### ¿Por qué overwrite?
En este pipeline, cada ejecución reprocesa todos los datos desde Bronze. El modo `overwrite` garantiza que la tabla Silver siempre refleje el estado más reciente de las transformaciones.

**Alternativas para producción**:
- `append` + deduplicación: Para procesamiento incremental
- `merge`: Para upserts basados en `loan_id`

### Metadata Agregada
- **_silver_timestamp**: Timestamp de cuándo se procesó el registro en Silver
- Útil para auditoría y tracking de procesamiento

### Características de los Datos en Silver:
✅ **Tipos de datos correctos**: Numéricos, enteros, strings
✅ **Datos validados**: Sin nulls en campos críticos, valores dentro de rangos
✅ **Datos estandarizados**: Campos categóricos normalizados
✅ **Listos para análisis**: Pueden usarse directamente en la capa Gold

In [0]:
from pyspark.sql.functions import current_timestamp

# ============================================================
# AGREGAR TIMESTAMP DE PROCESAMIENTO
# ============================================================
# Añadir columna con fecha/hora de cuándo se procesó en Silver
df_silver = df_standar.withColumn("_silver_timestamp", current_timestamp())

# ============================================================
# GUARDAR EN TABLA SILVER
# ============================================================
# Tabla destino: credit_risk_platform.silver.loans
# Modo: overwrite (reemplazar datos existentes)
# Formato: Delta Lake

table_name = "credit_risk_platform.silver.loans"

df_silver.write \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("="*60)
print("✅ DATOS GUARDADOS EXITOSAMENTE EN SILVER")
print("="*60)
print(f"\n💾 Tabla: {table_name}")
print(f"📊 Total de registros escritos: {df_silver.count():,}")
print(f"🔢 Tipos de datos: Convertidos y validados")
print(f"🔄 Estandarización: Aplicada a campos categóricos")
print("\n" + "="*60)

---
## ✔️ Paso 6: Validar Tabla Silver

Verificar que los datos se guardaron correctamente y consultar la tabla Silver.

In [0]:
%sql
-- ============================================================
-- VALIDAR CANTIDAD DE REGISTROS EN TABLA SILVER
-- ============================================================
-- Verificar que los datos se guardaron correctamente

SELECT COUNT(*) as total_loans
FROM credit_risk_platform.silver.loans